# Notebook 3 — Context & risk-factor layers

The flood mask tells us where it flooded *once* (July 2023). To make a **risk map** that says where flooding is *likely in general*, we need the underlying ingredients that make a place flood-prone. This notebook builds them, all snapped to the **exact same grid** as the flood mask so they stack perfectly.

Layers we'll create (saved to `data/outputs/`):

| Layer | Why it matters for flood risk |
|---|---|
| **Elevation** (DEM) | Water flows downhill and pools in low spots |
| **Slope** | Flat ground drains slowly → water sits |
| **Distance to river** | Closer to the Yamuna = more exposed |
| **Built-up** | Concrete can't absorb water → more runoff |

Plus the **road network**, which is what we ultimately want to flag for motorists.

## Step 1 — Setup and the reference grid

We load `flood_mask.tif` once and treat it as the **reference grid** — the master template that every other layer must match (same size, same pixel positions). We grab two views of it: one with `rioxarray` (used to *align* downloaded layers) and one with `rasterio` (used to get the exact transform for rasterising).

In [1]:
import ee, geemap
import numpy as np
import rasterio
import rioxarray as rxr
import geopandas as gpd
import osmnx as ox
from rasterio.features import rasterize
from rasterio.enums import Resampling
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt
from pathlib import Path

ee.Initialize(project="urban-flood-analysis-ncr-in")

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO / "data" / "raw"
OUT = REPO / "data" / "outputs"

# AOI (same box as Notebook 1)
W, S, E, N = 77.18, 28.50, 77.38, 28.78
aoi = ee.Geometry.Rectangle([W, S, E, N])

# Reference grid
ref = rxr.open_rasterio(OUT / "flood_mask.tif").squeeze()
with rasterio.open(OUT / "flood_mask.tif") as src:
    ref_transform = src.transform
    ref_shape = (src.height, src.width)
    ref_crs = src.crs
    ref_profile = src.profile

# Pixel size in metres (grid is in degrees, so convert at the AOI's latitude)
lat_mid = (S + N) / 2
px_x_m = abs(ref_transform.a) * 111_320 * np.cos(np.deg2rad(lat_mid))
px_y_m = abs(ref_transform.e) * 111_320
print("Reference grid:", ref_shape, "| ~pixel size (m):", round(px_x_m, 1), round(px_y_m, 1))

Reference grid: (3118, 2228) | ~pixel size (m): 8.8 10.0


In [2]:
def download_and_align(ee_image, raw_name, scale, resampling=Resampling.bilinear):
    """Download an EE image, then snap it onto the reference grid."""
    p = RAW / raw_name
    geemap.download_ee_image(ee_image, str(p), scale=scale, region=aoi, crs="EPSG:4326")
    r = rxr.open_rasterio(p).squeeze()
    return r.rio.reproject_match(ref, resampling=resampling)

## Step 2 — Elevation (DEM)

A **DEM** (Digital Elevation Model) gives the ground height in metres for every pixel. We use **Copernicus GLO-30** (30 m global elevation). Low-lying pixels are the natural collecting points for floodwater, so elevation is our single most important risk ingredient.

We download it, then `reproject_match` resamples it from 30 m up to our 10 m grid so it aligns with everything else.

In [3]:
dem_img = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM").mosaic().clip(aoi)
dem = download_and_align(dem_img, "dem_raw.tif", scale=30, resampling=Resampling.bilinear)
dem.rio.to_raster(OUT / "dem.tif")

dem_m = dem.values.astype("float32")
print("Elevation range (m):", round(float(np.nanmin(dem_m)), 1), "to", round(float(np.nanmax(dem_m)), 1))

/Users/user/miniconda3/envs/yamuna-flood/lib/python3.11/site-packages/geemap/common.py:12434: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


  0%|          |0/2 tiles [00:00<?]

/Users/user/miniconda3/envs/yamuna-flood/lib/python3.11/site-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'None'.
  return STACClient().get(self.id)


Elevation range (m): 182.8 to 272.1


## Step 3 — Slope

**Slope** is how steep the ground is, in degrees. We compute it from the DEM: `numpy.gradient` measures how fast elevation changes left-to-right and top-to-bottom, and we combine those into a steepness angle. Flat areas (low slope) drain slowly and let water accumulate, so flat = higher risk.

In [4]:
dz_dy, dz_dx = np.gradient(dem_m, px_y_m, px_x_m)
slope = np.degrees(np.arctan(np.sqrt(dz_dx ** 2 + dz_dy ** 2))).astype("float32")

prof = ref_profile.copy()
prof.update(dtype="float32", nodata=None, compress="lzw")
with rasterio.open(OUT / "slope.tif", "w", **prof) as dst:
    dst.write(slope, 1)
print("Slope range (deg):", round(float(slope.min()), 2), "to", round(float(slope.max()), 2))

Slope range (deg): 0.0 to 38.55


## Step 4 — Built-up surface

**ESA WorldCover** is a 10 m land-cover map. Class `50` is *built-up* (buildings, roads, concrete). Impervious surfaces can't soak up rain, so water runs off fast and overwhelms drains — built-up areas carry more urban-flood risk. We download just the built-up yes/no layer and align it (using *nearest* resampling, since it's categories, not a smooth surface).

In [5]:
wc = ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi)
builtup_img = wc.eq(50).rename("builtup")
builtup = download_and_align(builtup_img, "builtup_raw.tif", scale=10, resampling=Resampling.nearest)
builtup.astype("uint8").rio.to_raster(OUT / "builtup.tif")
print("Built-up fraction of AOI:", round(float(builtup.values.mean()) * 100, 1), "%")

/Users/user/miniconda3/envs/yamuna-flood/lib/python3.11/site-packages/geemap/common.py:12434: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


  0%|          |0/7 tiles [00:00<?]

Built-up fraction of AOI: 56.1 %


## Step 5 — Distance to the river

We pull the Yamuna (and nearby waterways) from OpenStreetMap as lines, **rasterise** them (paint the river onto a blank copy of our grid), then use a **distance transform** — a tool that fills every pixel with how far it is from the nearest painted river pixel. Closer to the river → higher flood exposure. We save the distance in metres.

In [6]:
rivers = ox.features_from_bbox(bbox=(W, S, E, N), tags={"waterway": "river"})
rivers = rivers[rivers.geom_type.isin(["LineString", "MultiLineString"])].to_crs(ref_crs)
rivers.to_file(OUT / "rivers.geojson", driver="GeoJSON")

river_raster = rasterize(
    [(geom, 1) for geom in rivers.geometry],
    out_shape=ref_shape, transform=ref_transform, fill=0, all_touched=True, dtype="uint8",
)
dist_m = (distance_transform_edt(river_raster == 0) * ((px_x_m + px_y_m) / 2)).astype("float32")

with rasterio.open(OUT / "dist_river.tif", "w", **prof) as dst:
    dst.write(dist_m, 1)
print(f"River segments: {len(rivers)} | distance range: 0 to {dist_m.max()/1000:.1f} km")

River segments: 15 | distance range: 0 to 14.8 km


## Step 6 — Major road network

Finally, the roads — the thing we want to warn drivers about. We grab the major drivable roads (motorway, trunk, primary, secondary) from OpenStreetMap and save them. In Notebook 4 we'll score each road by the flood risk underneath it.

In [7]:
cf = '["highway"~"motorway|trunk|primary|secondary"]'
G = ox.graph_from_bbox(bbox=(W, S, E, N), network_type="drive", custom_filter=cf,
                       simplify=True, retain_all=True)
roads = ox.graph_to_gdfs(G, nodes=False).to_crs(ref_crs)
roads = roads[["highway", "geometry"] + (["name"] if "name" in roads.columns else [])].reset_index(drop=True)
roads["highway"] = roads["highway"].astype(str)
roads.to_file(OUT / "roads.geojson", driver="GeoJSON")
print("Saved", len(roads), "major road segments.")

Saved 4911 major road segments.


## Step 7 — Look at all the layers

A quick visual check that each layer looks sensible and lines up with the others.

In [8]:
fig, ax = plt.subplots(1, 4, figsize=(20, 7))
for a, data, title, cmap in [
    (ax[0], dem_m, "Elevation (m)", "terrain"),
    (ax[1], slope, "Slope (deg)", "magma"),
    (ax[2], dist_m / 1000, "Distance to river (km)", "viridis_r"),
    (ax[3], builtup.values, "Built-up (1=yes)", "Greys"),
]:
    im = a.imshow(data, cmap=cmap)
    a.set_title(title)
    a.axis("off")
    fig.colorbar(im, ax=a, shrink=0.6)
plt.tight_layout()
plt.show()

/var/folders/jr/0tppzf_n6rscbz7qzblfgwbc0000gn/T/ipykernel_14677/283813656.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Recap

You now have four risk-factor rasters (elevation, slope, distance-to-river, built-up) all on the same grid as the flood mask, plus the major road network. 

**Next — Notebook 4:** combine these into a single **flood-risk score** for every pixel, rank the roads by risk, and make the final maps.